# RAG Exploration — P6 Research Agent

Empirical tuning of chunk size, overlap, and retrieval k. Run before wiring the vector store into the agent.

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.tools.vector_store import index, search

## Seed corpus

In [ ]:
DOCS = [
    {"text": "TP53 encodes the tumour suppressor protein p53. It is the most frequently mutated gene in human cancers. p53 activates DNA repair, arrests the cell cycle, and triggers apoptosis.", "source": "tp53_overview"},
    {"text": "TP53 mutations in glioblastoma multiforme occur in 25-30% of cases. R175H and R273H are the most common hotspot mutations.", "source": "tp53_gbm"},
    {"text": "BRCA1 and BRCA2 are tumour suppressor genes involved in homologous recombination DNA repair. PARP inhibitors exploit synthetic lethality in BRCA-deficient tumours.", "source": "brca_overview"},
    {"text": "KRAS G12C, G12D, and G12V mutations lock the protein in a GTP-bound active state. Sotorasib selectively inhibits KRAS G12C.", "source": "kras_overview"},
    {"text": "Checkpoint blockade with anti-PD-1 and anti-CTLA-4 antibodies has transformed treatment of melanoma and NSCLC. Response correlates with tumour mutational burden.", "source": "immunotherapy_overview"},
    {"text": "SciLifeLab provides national genomics infrastructure for Swedish researchers, including whole-genome sequencing and single-cell RNA-seq.", "source": "scilifelab_overview"},
]

## Baseline retrieval (chunk_size=512, overlap=64, k=5)

In [ ]:
with tempfile.TemporaryDirectory() as d:
    n = index(DOCS, persist_dir=d)
    print(f'Indexed {n} chunks\n')
    for q in ['tumour suppressor gene', 'DNA repair BRCA', 'KRAS inhibitor', 'genomics Sweden']:
        results = search(q, k=3, persist_dir=d)
        print(f"Query: '{q}'")
        for r in results:
            print(f"  [{r['score']:.3f}] ({r['source']}) {r['text'][:70]}")
        print()

## Chunk size sweep (256 / 512 / 1024)

Smaller chunks → more precise match, less context per result. Larger chunks → more context, lower similarity scores.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

MODEL = SentenceTransformer('all-MiniLM-L6-v2')
QUERY = 'TP53 mutation cancer'
q_emb = MODEL.encode([QUERY]).tolist()

def _index_custom(docs, chunk_size, overlap, d):
    client = chromadb.PersistentClient(path=d)
    coll = client.get_or_create_collection('rag')
    ids, texts, metas = [], [], []
    for doc in docs:
        start, i = 0, 0
        while start < len(doc['text']):
            chunk = doc['text'][start:start+chunk_size]
            ids.append(f"{doc['source']}_{i}")
            texts.append(chunk)
            metas.append({'source': doc['source']})
            start += chunk_size - overlap
            i += 1
    embs = MODEL.encode(texts).tolist()
    coll.upsert(ids=ids, embeddings=embs, documents=texts, metadatas=metas)
    return client, coll, len(ids)

for cs in [256, 512, 1024]:
    with tempfile.TemporaryDirectory() as d:
        _, coll, n = _index_custom(DOCS, cs, 64, d)
        res = coll.query(query_embeddings=q_emb, n_results=min(3, n))
        print(f'chunk_size={cs}: {n} chunks')
        for doc, dist in zip(res['documents'][0], res['distances'][0]):
            print(f'  [{1-dist:.3f}] {doc[:70]}')
        print()

## k sweep (1 / 3 / 5)

Higher k = more context for the LLM, but more tokens per prompt.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    index(DOCS, persist_dir=d)
    for k in [1, 3, 5]:
        results = search('TP53 cancer mutation', k=k, persist_dir=d)
        print(f'k={k}: {len(results)} results')
        for r in results:
            print(f"  [{r['score']:.3f}] ({r['source']})")
        print()

## Chosen parameters

| Parameter | Tested | Chosen | Reason |
|-----------|--------|--------|--------|
| chunk_size | 256, 512, 1024 | 512 | Best balance of precision and context for abstract-length text |
| overlap | 64 | 64 | Prevents semantic loss at chunk boundaries |
| k | 1, 3, 5 | 5 | Enough context without saturating the LLM prompt |
| model | all-MiniLM-L6-v2 | all-MiniLM-L6-v2 | Small, fast, effective for scientific text |